In [11]:
samples = [True,False,False]
all(samples)
import numpy as np
import pandas as pd
samples = np.array(['사과','바나나','딸기','사과','오렌지','바나나','바나나'])
df = pd.DataFrame(samples,columns=['과일'])
df['과일'].mode()

0    바나나
Name: 과일, dtype: object

In [12]:
# 결측치 탐지
import pandas as pd
import numpy as np

df = pd.read_csv(r'data\messy_data.csv')
print(f'원본 데이터 shape : { df.shape}')
print(f'결측치 개수 : { df.isnull().sum().sum()}')
print('='*50)
print(f'결측치 비율 : { df.isnull().mean().round(2)}')
print(f'결측치가 포함된 행의개수 : { df.isnull().any(axis=1).sum()}')

원본 데이터 shape : (112, 6)
결측치 개수 : 24
결측치 비율 : id            0.00
name          0.00
age           0.06
salary        0.05
score         0.10
department    0.00
dtype: float64
결측치가 포함된 행의개수 : 22


In [13]:
df.tail().isnull().astype(int)

,id,name,age,salary,score,department
107,0,0,0,0,1,0
108,0,0,0,0,0,0
109,0,0,0,0,0,0
110,0,0,0,0,0,0
111,0,0,0,0,0,0


In [14]:
# 결측치 삭제
# 결측치가 있는 행만 출력
df[df.isna().any(axis=1)]
# 결측치가 하나라도 있는 행을 삭제
df_drop_any = df.dropna(how='any')
# 모든 값이 NaN인 행만 삭제
df_drop_all = df.dropna(how='all')
len(df_drop_all) - len(df)
# 특정컬럼기준 삭제
df.dropna(subset=['age','salary'])
# thresh - 최소 N개 유효값이 있어야 보존
df_drop_thresh = df.dropna(thresh=4)
df_drop_thresh[df_drop_thresh.isna().any(axis=1)]

,id,name,age,salary,score,department
5,6,user_6,NaN,58956.0,96.54,개발
6,7,user_7,300.0,NaN,NaN,영업
8,9,user_9,40.0,72034.0,NaN,개발
12,13,user_13,53.0,56868.0,NaN,영업
14,15,user_15,NaN,54903.0,88.78,인사
15,16,user_16,20.0,48783.0,NaN,인사
21,22,user_22,55.0,NaN,94.01,마케팅
29,30,user_30,44.0,NaN,1.81,마케팅
35,36,user_36,61.0,45258.0,NaN,재무
40,41,user_41,26.0,38389.0,NaN,개발


In [15]:
# 결측치 대치
# 고정값으로 대치
df_fill_zero =  df.copy()
temp = df_fill_zero.isna().any()
print(temp[temp.values].index)
df_fill_zero['score'].fillna( df_fill_zero['score'].median() )
dept_mode = df_fill_zero['department'].mode()[0]
df_fill_zero['age'].bfill()

Index(['age', 'salary', 'score'], dtype='object')


0      56.0
1      46.0
2      32.0
3      60.0
4      25.0
       ... 
107    38.0
108    28.0
109    56.0
110    41.0
111    59.0
Name: age, Length: 112, dtype: float64

In [16]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/messy_data.csv')
df['score'].interpolate(method='polynomial',order=3)

0       17.500000
1       98.220000
2       51.660000
3       26.080000
4       99.630000
          ...    
107    118.594571
108     70.040000
109     17.500000
110     66.900000
111     49.390000
Name: score, Length: 112, dtype: float64

In [17]:
import pandas as pd
import numpy as np
from glob import glob
df = pd.read_csv(glob('../data/*.csv')[1],low_memory=False)

In [18]:
len(df.columns)

39

In [19]:
# 이상치 탐지
%pwd
import pandas as pd
import numpy as np
df = pd.read_csv(r'data\messy_data.csv')
numeric_cols = df.describe().columns
# 이상치 탐지를 위해서 결측치를 중앙값으로 대처
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [20]:
# IQR 
def detect_outlier_iqr(data,column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)
    outlier_mask = (data[column] > upper ) | (data[column] < lower)
    return outlier_mask,lower,upper,Q1,Q3,IQR

In [21]:
numeric_cols = df.describe().columns[ 1: ]

iqr_df = df.copy()
total_removed_counts = 0
for colname in numeric_cols:    
    outlier_mask,lower,upper,Q1,Q3,IQR =  detect_outlier_iqr(iqr_df,colname)
    print(f'컬럼명 : {colname}')
    print(f'이상치 개수 : {outlier_mask.sum()}')
    print('-'*50)
    before = len(iqr_df)
    iqr_df = iqr_df[~outlier_mask]
    removed = before - len(iqr_df)
    total_removed_counts += removed

print(f'IQR 제거후 shape {iqr_df.shape}')
print(f'총 제거수 : {total_removed_counts}')



컬럼명 : age
이상치 개수 : 4
--------------------------------------------------
컬럼명 : salary
이상치 개수 : 6
--------------------------------------------------
컬럼명 : score
이상치 개수 : 0
--------------------------------------------------
IQR 제거후 shape (102, 6)
총 제거수 : 10


In [23]:
# 데이터가 정규분포를 따를다고 가정
from scipy import stats
# (각 요소 - 평균)/ 표준편차
stats.zscore(df['age']) > 3
def detect_outlier_zscore(data, colname, thereshold=3):
    z_score = np.abs( stats.zscore(data[colname].dropna()) )
    return z_score > thereshold

In [25]:
for col in numeric_cols:
    before = len(df)
    zscore_index = detect_outlier_zscore(df,col)
    df = df[zscore_index]
    removed = before - len(df)
    print(col, removed)

age 109
salary 3
score 0


In [58]:
# 이상치 탐색(수치형 데이터)
# 원본이 아닌 복사본으로 결측치, 이상치 탐색, 대처
# IQR
# Z-score
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('data/아파트(매매)_실거래가_20260403104248.csv', encoding='cp949', header=15)
df = pd.DataFrame(df)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',','').astype(int)
print(df)
df.describe()

        NO             시군구      번지   본번  부번                       단지명  \
0        1   서울특별시 강남구 역삼동  720-25  720  25                     대우디오빌   
1        2   서울특별시 강남구 도곡동   543-7  543   7                  도곡1차아이파크   
2        3   서울특별시 강남구 도곡동     952  952   0                     대우디오빌   
3        4   서울특별시 강남구 청담동      42   42   0                   삼성청담아파트   
4        5   서울특별시 강남구 수서동     736  736   0                       신동아   
...    ...             ...     ...  ...  ..                       ...   
2963  2964  서울특별시 강남구 압구정동     456  456   0  현대6차(78~81,83,84,86,87동)   
2964  2965   서울특별시 강남구 대치동     511  511   0                   한보미도맨션2   
2965  2966   서울특별시 강남구 대치동     506  506   0               선경1차(1동-7동)   
2966  2967   서울특별시 강남구 대치동     316  316   0                        은마   
2967  2968   서울특별시 강남구 도곡동     527  527   0                      도곡렉슬   

       전용면적(㎡)    계약년월  계약일  거래금액(만원)    동   층 매수자 매도자  건축년도        도로명  \
0      30.0300  202603   31     27000    -  17  

,NO,본번,부번,전용면적(㎡),계약년월,계약일,거래금액(만원),층,건축년도
count,2968.000000,2968.000000,2968.000000,2968.000000,2968.000000,2968.000000,2.968000e+03,2968.000000,2968.000000
mean,1484.500000,538.760782,3.614218,86.399945,202521.773922,16.478437,2.879723e+05,9.196765,2000.747305
std,856.932125,355.302426,9.308358,39.168293,33.462621,9.043665,1.886570e+05,6.884455,12.659291
min,1.000000,2.000000,0.000000,12.449000,202504.000000,1.000000,1.350000e+04,1.000000,1976.000000
25%,742.750000,187.000000,0.000000,59.847500,202506.000000,9.000000,1.610000e+05,4.000000,1992.000000
50%,1484.500000,579.000000,0.000000,84.460000,202508.000000,17.000000,2.550000e+05,8.000000,2002.000000
75%,2226.250000,746.000000,1.000000,107.380000,202511.000000,25.000000,3.592500e+05,12.000000,2012.000000
max,2968.000000,1284.000000,74.000000,273.960000,202603.000000,31.000000,1.900000e+06,64.000000,2025.000000


In [55]:
1.900000e+06

1900000.0

In [65]:
# 이상치가 아닌 clean area에 해당하는 시군구
# 이상치로 판단되는 시군구
df.describe().columns
df_copy = df.copy()
df_copy['거래금액(만원)']

def outlier(data, colname):
    Q1 = data[colname].quantile(0.25)
    Q3 = data[colname].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - (1.5*IQR)
    upper = Q3 + (1.5*IQR)
    outlier_mask = (data[colname] > upper) | (data[colname] < lower)
    return outlier_mask, upper, lower

outlier_index, upper, lower = outlier(df_copy, '거래금액(만원)')
df[outlier_index]['시군구'].unique(), lower, upper


(array(['서울특별시 강남구 압구정동', '서울특별시 강남구 삼성동', '서울특별시 강남구 청담동',
        '서울특별시 강남구 신사동', '서울특별시 강남구 대치동', '서울특별시 강남구 도곡동'], dtype=object),
 np.float64(-136375.0),
 np.float64(656625.0))

In [88]:
import os 
import pandas as pd

data_path = 'data/messy_data.csv'
df = pd.read_csv(data_path)

duplicates_all = df.duplicated()
num_duplicates = duplicates_all.sum()

if num_duplicates > 0:
    print(df[df.duplicated(keep=False)].sort_values(by='id'))

     id     name   age   salary  score department
0     1   user_1  56.0  59545.0  17.50        마케팅
109   1   user_1  56.0  59545.0  17.50        마케팅
108  11  user_11  28.0  64250.0  70.04         인사
10   11  user_11  28.0  64250.0  70.04         인사
18   19  user_19  41.0  38304.0  66.90         인사
110  19  user_19  41.0  38304.0  66.90         인사
106  23  user_23  19.0  52257.0  97.37         인사
22   23  user_23  19.0  52257.0  97.37         인사
30   31  user_31  59.0  74344.0  49.39         인사
111  31  user_31  59.0  74344.0  49.39         인사
105  40  user_40  38.0  46855.0  25.05         영업
39   40  user_40  38.0  46855.0  25.05         영업
44   45  user_45  42.0  62749.0  43.44         영업
104  45  user_45  42.0  62749.0  43.44         영업
103  46  user_46  31.0  30330.0  35.01         재무
45   46  user_46  31.0  30330.0  35.01         재무
53   54  user_54  61.0  42293.0   4.36         개발
101  54  user_54  61.0  42293.0   4.36         개발
70   71  user_71  51.0  38561.0  39.47         개발


In [81]:
dup_by_id = df.duplicated(subset=['id']).sum()

dup_by_name_dept = df.duplicated(subset=['name', 'department'])
df[dup_by_name_dept]

,id,name,age,salary,score,department
100,84,user_84,NaN,56472.0,55.28,인사
101,54,user_54,61.0,42293.0,4.36,개발
102,71,user_71,51.0,38561.0,39.47,개발
103,46,user_46,31.0,30330.0,35.01,재무
104,45,user_45,42.0,62749.0,43.44,영업
105,40,user_40,38.0,46855.0,25.05,영업
106,23,user_23,19.0,52257.0,97.37,인사
107,81,user_81,38.0,50421.0,NaN,영업
108,11,user_11,28.0,64250.0,70.04,인사
109,1,user_1,56.0,59545.0,17.50,마케팅


In [89]:
df_keep_first = df.drop_duplicates(keep='first')
print(f"\n▶ keep='first' 적용 시 Shape: {df.shape} -> {df_keep_first.shape}")
print(f"   삭제된 행 수: {len(df) - len(df_keep_first)}")


▶ keep='first' 적용 시 Shape: (112, 6) -> (100, 6)
   삭제된 행 수: 12


In [82]:
df_keep_last = df.drop_duplicates(keep='last')
print(f"\n▶ keep='last' 적용 시 Shape: {df.shape} -> {df_keep_last.shape}")
print(f"   삭제된 행 수: {len(df) - len(df_keep_last)}")


▶ keep='last' 적용 시 Shape: (112, 6) -> (100, 6)
   삭제된 행 수: 12


In [83]:
# keep=False : 중복이 존재하는 모든 행을 일괄 삭제 (고유한 값만 남김)
df_keep_none = df.drop_duplicates(keep=False)
print(f"\n▶ keep=False 적용 시 Shape: {df.shape} -> {df_keep_none.shape}")
print(f"   삭제된 행 수: {len(df) - len(df_keep_none)}")


▶ keep=False 적용 시 Shape: (112, 6) -> (88, 6)
   삭제된 행 수: 24


In [84]:
df_unique_id = df.drop_duplicates(subset=['id'], keep='last')
print(f"\n▶ subset=['id'], keep='last' 적용 시 Shape: {df.shape} -> {df_unique_id.shape}")
print(f"   삭제된 행 수: {len(df) - len(df_unique_id)}")


▶ subset=['id'], keep='last' 적용 시 Shape: (112, 6) -> (100, 6)
   삭제된 행 수: 12


In [ ]:
df_final = df_keep_first.copy()

